# Notebook 10 — Model v3: Formation-Stratified RF
**Project PaleoWave** | Phase 3

Terrain rasters (slope/aspect/TRI/TPI) are re-derived from `dem_merged.tif` using richdem,
written as BigTIFF, then used to train formation-stratified north/south submodels.

- **North**: Prida + Favret (≥39.5°N) — Random Forest
- **South**: Luning + Gabbs (<39.5°N) — RF or SVM, best LOO wins
- **TPI**: direct RF feature (not post-hoc rule as in v2)


In [1]:
import pandas as pd
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.transform import rowcol
from scipy.ndimage import uniform_filter, generic_filter
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
import joblib
import warnings
warnings.filterwarnings('ignore')

BASE       = Path(r'C:\Users\brook\Documents\Project-PaleoWave')
DEM_PATH   = BASE / 'data' / 'dem'     / 'dem_merged.tif'
TERRAIN_DIR= BASE / 'data' / 'terrain'
PBDB_DIR   = BASE / 'data' / 'pbdb'
MODEL_DIR  = BASE / 'data' / 'model'

SPLIT_LAT    = 39.5
FEATURE_COLS = ['elevation_m', 'slope_deg', 'aspect_deg', 'tri', 'tpi']
RF_PARAMS    = dict(n_estimators=500, max_depth=6,
                    class_weight='balanced', random_state=42, n_jobs=-1)

print('Imports OK.')
with rasterio.open(DEM_PATH) as src:
    print(f'DEM: {src.count} band  crs={src.crs}  nodata={src.nodata}')
    print(f'     shape={src.shape}  bounds={src.bounds}')


Imports OK.
DEM: 1 band  crs=EPSG:4269  nodata=-999999.0
     shape=(32412, 21612)  bounds=BoundingBox(left=-119.0005555565931, bottom=37.99944443915925, right=-116.99944444332935, top=41.000555556194854)


In [2]:
## 2. Derive Terrain Features — Subsample to 90m first, then compute

print('Reading DEM...')
with rasterio.open(DEM_PATH) as src:
    full_shape = src.shape
    nodata     = src.nodata
    full_res_x = abs(src.transform.a)
    lat_mid    = (src.bounds.top + src.bounds.bottom) / 2
    full_profile = src.profile.copy()
    
    # Subsample factor: target ~90m, current ~8m → factor ~11
    m_per_deg_lon = 111320.0 * np.cos(np.radians(lat_mid))
    cell_m = full_res_x * m_per_deg_lon
    target_m = 90.0
    factor = max(1, round(target_m / cell_m))
    
    print(f'Full resolution: {cell_m:.1f}m/px  shape={full_shape}')
    print(f'Subsampling by {factor}x → ~{cell_m*factor:.0f}m/px')
    
    # Read with decimation
    out_shape = (1,
                 full_shape[0] // factor,
                 full_shape[1] // factor)
    elev = src.read(1,
                    out_shape=out_shape[1:],
                    resampling=rasterio.enums.Resampling.average
                   ).astype(np.float32)
    
    # Recalculate transform for subsampled raster
    from rasterio.transform import from_bounds
    sub_transform = from_bounds(
        src.bounds.left, src.bounds.bottom,
        src.bounds.right, src.bounds.top,
        elev.shape[1], elev.shape[0]
    )

print(f'Subsampled shape: {elev.shape}')

# Mask nodata
nd_mask = (elev == nodata) | ~np.isfinite(elev)
elev[nd_mask] = np.nan

cell_size_x = (90.0 if factor > 1 else cell_m)
cell_size_y = cell_size_x
print(f'Working cell size: ~{cell_size_x:.0f}m')

# --- Slope & Aspect ---
print('Computing slope/aspect...')
dz_dy, dz_dx = np.gradient(np.nan_to_num(elev, nan=0.0),
                             cell_size_y, cell_size_x)
slope  = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))
aspect = np.degrees(np.arctan2(-dz_dx, dz_dy)) % 360
slope[nd_mask]  = np.nan
aspect[nd_mask] = np.nan
print(f'  Slope:  {np.nanmin(slope):.1f}° – {np.nanmax(slope):.1f}°')
print(f'  Aspect: {np.nanmin(aspect):.1f}° – {np.nanmax(aspect):.1f}°')

# --- TPI ---
print('Computing TPI...')
e_filled  = np.nan_to_num(elev, nan=0.0)
elev_mean = uniform_filter(e_filled, size=3)
tpi       = elev - elev_mean
tpi[nd_mask] = np.nan
print(f'  TPI: {np.nanmin(tpi):.1f} – {np.nanmax(tpi):.1f}  mean={np.nanmean(tpi):.3f}')

# --- TRI (vectorized) ---
print('Computing TRI...')
e = np.nan_to_num(elev, nan=0.0)
tri_inner = (
    np.abs(e[1:-1,1:-1] - e[0:-2,0:-2]) +
    np.abs(e[1:-1,1:-1] - e[0:-2,1:-1]) +
    np.abs(e[1:-1,1:-1] - e[0:-2,2:  ]) +
    np.abs(e[1:-1,1:-1] - e[1:-1,0:-2]) +
    np.abs(e[1:-1,1:-1] - e[1:-1,2:  ]) +
    np.abs(e[1:-1,1:-1] - e[2:  ,0:-2]) +
    np.abs(e[1:-1,1:-1] - e[2:  ,1:-1]) +
    np.abs(e[1:-1,1:-1] - e[2:  ,2:  ])
) / 8.0
tri = np.full_like(elev, np.nan)
tri[1:-1, 1:-1] = tri_inner
tri[nd_mask] = np.nan
print(f'  TRI: {np.nanmin(tri):.2f} – {np.nanmax(tri):.2f}')

print('\nAll terrain features computed.')

# Update profile for subsampled output
sub_profile = full_profile.copy()
sub_profile.update(
    count=1, dtype='float32', nodata=-9999.0,
    width=elev.shape[1], height=elev.shape[0],
    transform=sub_transform,
    compress='lzw', BIGTIFF='YES'
)

Reading DEM...
Full resolution: 8.0m/px  shape=(32412, 21612)
Subsampling by 11x → ~87m/px
Subsampled shape: (2946, 1964)
Working cell size: ~90m
Computing slope/aspect...
  Slope:  0.0° – 57.5°
  Aspect: 0.0° – 360.0°
Computing TPI...
  TPI: -68.1 – 76.6  mean=-0.000
Computing TRI...
  TRI: 0.00 – 105.66

All terrain features computed.


In [4]:
## 3. Write Terrain Rasters as BigTIFF

def write_raster(arr, path, profile):
    out = np.where(np.isnan(arr), -9999.0, arr).astype(np.float32)
    with rasterio.open(path, 'w', **profile) as dst:
        dst.write(out, 1)
    print(f'  Written: {path.name}  ({path.stat().st_size/1e6:.0f} MB)')

for fname, arr in [('slope_v3.tif',   slope),
                   ('aspect_v3.tif',  aspect),
                   ('tri_v3.tif',     tri),
                   ('tpi_v3.tif',     tpi),
                   ('elevation_v3.tif', elev)]:
    write_raster(arr, TERRAIN_DIR / fname, sub_profile)

feature_rasters = {
    'elevation_m': TERRAIN_DIR / 'elevation_v3.tif',
    'slope_deg'  : TERRAIN_DIR / 'slope_v3.tif',
    'aspect_deg' : TERRAIN_DIR / 'aspect_v3.tif',
    'tri'        : TERRAIN_DIR / 'tri_v3.tif',
    'tpi'        : TERRAIN_DIR / 'tpi_v3.tif',
}
print('\nFeature rasters ready:')
for k, v in feature_rasters.items():
    print(f'  {k:15s}: {v.name}')

  Written: slope_v3.tif  (27 MB)
  Written: aspect_v3.tif  (26 MB)
  Written: tri_v3.tif  (23 MB)
  Written: tpi_v3.tif  (20 MB)
  Written: elevation_v3.tif  (24 MB)

Feature rasters ready:
  elevation_m    : elevation_v3.tif
  slope_deg      : slope_v3.tif
  aspect_deg     : aspect_v3.tif
  tri            : tri_v3.tif
  tpi            : tpi_v3.tif


In [5]:
## 4. Sample Terrain Features at PBDB Localities

def sample_raster(raster_path, lats, lons):
    values = []
    with rasterio.open(raster_path) as src:
        nd = src.nodata
        for lat, lon in zip(lats, lons):
            try:
                row, col = src.index(lon, lat)
                val = float(src.read(1, window=Window(col, row, 1, 1))[0, 0])
                if (nd is not None and val == nd) or not np.isfinite(val):
                    val = np.nan
            except Exception:
                val = np.nan
            values.append(val)
    return values

# Load PBDB and infer missing formations
pbdb = pd.read_csv(PBDB_DIR / 'pbdb_occurrences_clean.csv')

def infer_formation(row):
    if pd.notna(row.formation): return row.formation
    member = str(row.get('member', '')).lower()
    strat  = str(row.get('stratgroup', '')).lower()
    if 'star peak' in strat or 'fossil hill' in member: return 'Favret'
    if 'prida'  in member: return 'Prida'
    if 'luning' in member or 'luning' in strat: return 'Luning'
    return 'Prida_inferred' if row.latitude >= SPLIT_LAT else 'Luning_inferred'

pbdb['formation_v3'] = pbdb.apply(infer_formation, axis=1)
print(f'PBDB: {len(pbdb)} records')
print(pbdb.formation_v3.value_counts().to_string())
print()

# Extract features
feats = pbdb[['occurrence_id', 'latitude', 'longitude', 'formation_v3']].copy()
for feat, rpath in feature_rasters.items():
    print(f'  Sampling {feat}...')
    feats[feat] = sample_raster(rpath, feats.latitude, feats.longitude)

print()
print(feats[['occurrence_id', 'formation_v3'] + FEATURE_COLS].to_string(index=False))
print()
print('NaN counts:')
print(feats[FEATURE_COLS].isna().sum().to_string())


PBDB: 30 records
formation_v3
Prida             10
Favret             7
Luning             6
Prida_inferred     6
Gabbs              1

  Sampling elevation_m...
  Sampling slope_deg...
  Sampling aspect_deg...
  Sampling tri...
  Sampling tpi...

occurrence_id   formation_v3  elevation_m  slope_deg  aspect_deg       tri        tpi
   occ:361674          Prida  1395.810425   6.660565  170.420334 10.184006  -4.848755
   occ:467069          Prida  1395.810425   6.660565  170.420334 10.184006  -4.848755
   occ:467070          Prida  1395.810425   6.660565  170.420334 10.184006  -4.848755
   occ:467071          Prida  1395.810425   6.660565  170.420334 10.184006  -4.848755
   occ:467073          Prida  1395.810425   6.660565  170.420334 10.184006  -4.848755
   occ:467074          Prida  1395.810425   6.660565  170.420334 10.184006  -4.848755
   occ:714514          Gabbs  1698.549438  15.296494  144.823883 16.734680  -3.918335
   occ:900174         Luning  1575.939331   2.352936  276.028259

In [6]:
## 5. Load Background — Add New Terrain Features

bg = pd.read_csv(PBDB_DIR / 'paleowave_background_proper.csv')
print(f'Background: {len(bg)} points  columns: {bg.columns.tolist()}')

# Re-sample all features from new rasters for consistency
for feat, rpath in feature_rasters.items():
    print(f'  Sampling {feat} for background...')
    bg[feat] = sample_raster(rpath, bg.latitude, bg.longitude)

# Fill NaNs (outside raster extent) with column median
for feat in FEATURE_COLS:
    n = bg[feat].isna().sum()
    if n:
        med = bg[feat].median()
        bg[feat] = bg[feat].fillna(med)
        print(f'  Filled {n} NaNs in {feat} with median {med:.2f}')

print(f'\nBackground NaN counts after fill:')
print(bg[FEATURE_COLS].isna().sum().to_string())

bg_north = bg[bg.latitude >= SPLIT_LAT].copy()
bg_south = bg[bg.latitude <  SPLIT_LAT].copy()
print(f'\nBackground north: {len(bg_north)}  south: {len(bg_south)}')


Background: 982 points  columns: ['elevation_m', 'slope_deg', 'aspect_deg', 'tri', 'latitude', 'longitude', 'label']
  Sampling elevation_m for background...
  Sampling slope_deg for background...
  Sampling aspect_deg for background...
  Sampling tri for background...
  Sampling tpi for background...
  Filled 523 NaNs in elevation_m with median 1610.54
  Filled 523 NaNs in slope_deg with median 3.35
  Filled 523 NaNs in aspect_deg with median 167.11
  Filled 582 NaNs in tri with median 4.29
  Filled 523 NaNs in tpi with median -0.02

Background NaN counts after fill:
elevation_m    0
slope_deg      0
aspect_deg     0
tri            0
tpi            0

Background north: 469  south: 513


In [7]:
## 6. Build Training Arrays

def build_Xy(presence_df, bg_df, feature_cols):
    p = presence_df[feature_cols].copy()
    b = bg_df[feature_cols].copy()
    combined = pd.concat([p, b], ignore_index=True)
    for col in feature_cols:
        med = combined[col].median()
        combined[col] = combined[col].fillna(med)
    y = np.array([1]*len(p) + [0]*len(b))
    X = combined[feature_cols].values
    return X, y

pres_north = feats[feats.latitude >= SPLIT_LAT].copy()
pres_south = feats[feats.latitude <  SPLIT_LAT].copy()

X_north, y_north = build_Xy(pres_north, bg_north, FEATURE_COLS)
X_south, y_south = build_Xy(pres_south, bg_south, FEATURE_COLS)

for name, X, y, pres in [('North', X_north, y_north, pres_north),
                          ('South', X_south, y_south, pres_south)]:
    print(f'{name}: {X.shape[0]} total  '
          f'({y.sum()} presence / {(y==0).sum()} background)')
    print(f'  formations: {pres.formation_v3.value_counts().to_dict()}')
    print(f'  classes in y: {np.unique(y)}')
    print()


North: 492 total  (23 presence / 469 background)
  formations: {'Prida': 10, 'Favret': 7, 'Prida_inferred': 6}
  classes in y: [0 1]

South: 520 total  (7 presence / 513 background)
  formations: {'Luning': 6, 'Gabbs': 1}
  classes in y: [0 1]



In [8]:
## 7. Train North RF + South RF and SVM

rf_north = RandomForestClassifier(**RF_PARAMS)
rf_north.fit(X_north, y_north)
print('North RF — feature importances:')
for f, i in sorted(zip(FEATURE_COLS, rf_north.feature_importances_),
                   key=lambda x: -x[1]):
    print(f'  {f:15s}: {i:.3f}')
print()

rf_south = RandomForestClassifier(**RF_PARAMS)
rf_south.fit(X_south, y_south)
print('South RF — feature importances:')
for f, i in sorted(zip(FEATURE_COLS, rf_south.feature_importances_),
                   key=lambda x: -x[1]):
    print(f'  {f:15s}: {i:.3f}')
print()

scaler_south = StandardScaler()
svm_south = SVC(kernel='rbf', class_weight='balanced',
                probability=True, random_state=42, C=1.0, gamma='scale')
svm_south.fit(scaler_south.fit_transform(X_south), y_south)
print('South SVM trained.')


North RF — feature importances:
  tri            : 0.390
  slope_deg      : 0.224
  tpi            : 0.200
  aspect_deg     : 0.105
  elevation_m    : 0.081

South RF — feature importances:
  tri            : 0.371
  tpi            : 0.228
  aspect_deg     : 0.153
  slope_deg      : 0.147
  elevation_m    : 0.101

South SVM trained.


In [9]:
## 8. LOO Validation

def loo_validate(presence_df, bg_df, feature_cols, model_type='rf'):
    pres = presence_df[feature_cols + ['occurrence_id']].copy()
    for col in feature_cols:
        pres[col] = pres[col].fillna(pres[col].median())
    bg_v = bg_df[feature_cols].copy()
    for col in feature_cols:
        bg_v[col] = bg_v[col].fillna(bg_v[col].median())
    bg_arr = bg_v.values

    results = []
    for i in range(len(pres)):
        held  = pres.iloc[[i]]
        train = pres.drop(pres.index[i])
        X_tr  = np.vstack([train[feature_cols].values, bg_arr])
        y_tr  = np.array([1]*len(train) + [0]*len(bg_arr))
        X_te  = held[feature_cols].values
        if model_type == 'rf':
            m    = RandomForestClassifier(**RF_PARAMS)
            m.fit(X_tr, y_tr)
            prob = m.predict_proba(X_te)[0, 1]
        else:
            sc   = StandardScaler()
            m    = SVC(kernel='rbf', class_weight='balanced',
                       probability=True, random_state=42, C=1.0, gamma='scale')
            m.fit(sc.fit_transform(X_tr), y_tr)
            prob = m.predict_proba(sc.transform(X_te))[0, 1]
        results.append({'occurrence_id': held.occurrence_id.values[0],
                        'prob': round(prob, 3),
                        'correct': int(prob >= 0.5)})
    df = pd.DataFrame(results)
    return df, df.correct.mean()

print('LOO — North RF...')
loo_n,    recall_n     = loo_validate(pres_north, bg_north, FEATURE_COLS, 'rf')
print(f'  {recall_n:.1%}  ({loo_n.correct.sum()}/{len(loo_n)})')
print(loo_n.to_string(index=False))
print()

print('LOO — South RF...')
loo_s_rf, recall_s_rf  = loo_validate(pres_south, bg_south, FEATURE_COLS, 'rf')
print(f'  {recall_s_rf:.1%}  ({loo_s_rf.correct.sum()}/{len(loo_s_rf)})')
print(loo_s_rf.to_string(index=False))
print()

print('LOO — South SVM...')
loo_s_svm,recall_s_svm = loo_validate(pres_south, bg_south, FEATURE_COLS, 'svm')
print(f'  {recall_s_svm:.1%}  ({loo_s_svm.correct.sum()}/{len(loo_s_svm)})')
print(loo_s_svm.to_string(index=False))
print()

south_type   = 'rf' if recall_s_rf >= recall_s_svm else 'svm'
south_recall = max(recall_s_rf, recall_s_svm)
model_south  = rf_south   if south_type == 'rf' else svm_south
scaler_s     = None       if south_type == 'rf' else scaler_south
print(f'South selected: {south_type.upper()}  recall={south_recall:.1%}')


LOO — North RF...
  65.2%  (15/23)
occurrence_id  prob  correct
   occ:361674 0.986        1
   occ:467069 0.986        1
   occ:467070 0.986        1
   occ:467071 0.986        1
   occ:467073 0.986        1
   occ:467074 0.986        1
   occ:906286 0.973        1
   occ:906287 0.973        1
   occ:906313 0.614        1
   occ:906314 0.614        1
   occ:908660 0.690        1
   occ:908661 0.309        0
   occ:909076 0.468        0
   occ:934002 0.176        0
  occ:1095497 0.111        0
  occ:1186493 0.396        0
  occ:1412645 0.690        1
  occ:1412647 0.690        1
  occ:1417529 0.114        0
  occ:1419019 0.399        0
  occ:1419160 0.184        0
  occ:1504406 0.634        1
  occ:1583679 0.977        1

LOO — South RF...
  71.4%  (5/7)
occurrence_id  prob  correct
   occ:714514 0.277        0
   occ:900174 0.662        1
   occ:900219 0.662        1
   occ:900302 0.016        0
   occ:908629 0.882        1
  occ:1419479 0.882        1
  occ:1419480 0.882        1

LO

In [10]:
## 9. Score v2 Candidates with v3 Models

v2 = pd.read_csv(MODEL_DIR / 'paleowave_v2_top50.csv')
full_path = MODEL_DIR / 'paleowave_v2_candidates_full.csv'
cands = pd.read_csv(full_path) if full_path.exists() else v2.copy()
print(f'Candidate pool: {len(cands)} points')
print(f'Columns: {cands.columns.tolist()}')

# Sample all terrain features fresh from new rasters
for feat, rpath in feature_rasters.items():
    print(f'  Sampling {feat} for candidates...')
    cands[feat] = sample_raster(rpath, cands.latitude, cands.longitude)
    cands[feat] = cands[feat].fillna(cands[feat].median())

cands_n = cands[cands.latitude >= SPLIT_LAT].copy()
cands_s = cands[cands.latitude <  SPLIT_LAT].copy()
print(f'Scoring: {len(cands_n)} north  {len(cands_s)} south')

if len(cands_n):
    cands_n['v3_score'] = rf_north.predict_proba(
        cands_n[FEATURE_COLS].values)[:, 1]
    cands_n['submodel'] = 'north'

if len(cands_s):
    X_s = cands_s[FEATURE_COLS].values
    if scaler_s is not None: X_s = scaler_s.transform(X_s)
    cands_s['v3_score'] = model_south.predict_proba(X_s)[:, 1]
    cands_s['submodel'] = 'south'

v3 = pd.concat([c for c in [cands_n, cands_s] if len(c)],
               ignore_index=True)
v3 = v3.sort_values('v3_score', ascending=False).reset_index(drop=True)
v3['v3_rank'] = v3.index + 1

show = ['v3_rank','latitude','longitude','v3_score','tpi','submodel']
show = [c for c in show if c in v3.columns]
print()
print('=== v3 TOP 20 ===')
print(v3[show].head(20).to_string(index=False))


Candidate pool: 60 points
Columns: ['rank', 'latitude', 'longitude', 'ml_probability', 'on_triassic', 'dist_to_triassic_m', 'formation_code', 'formation_name', 'rock_type', 'geo_bonus', 'composite_score', 'priority', 'elevation_m', 'slope_deg', 'aspect_deg', 'tri', 'tpi_15m', 'tpi_adj', 'composite_v2', 'rank_v2', 'rank_v1', 'rank_delta']
  Sampling elevation_m for candidates...
  Sampling slope_deg for candidates...
  Sampling aspect_deg for candidates...
  Sampling tri for candidates...
  Sampling tpi for candidates...
Scoring: 39 north  21 south

=== v3 TOP 20 ===
 v3_rank  latitude   longitude  v3_score       tpi submodel
       1 40.265278 -117.479722  0.688830 -2.762207    north
       2 40.907222 -118.464444  0.659855 -2.090210    north
       3 40.404722 -118.243889  0.607730 18.208130    north
       4 40.095556 -117.245556  0.594901  3.918579    north
       5 40.095556 -117.245556  0.594901  3.918579    north
       6 40.406389 -117.705556  0.553001  0.860840    north
       

In [11]:
## 10. v2 vs v3 Comparison

score_col = next((c for c in v2.columns if 'composite' in c.lower()), None)
rank_col  = next((c for c in v2.columns if 'rank'      in c.lower()), None)

if score_col:
    print(f'=== v2 TOP-10 → v3 (score={score_col}) ===')
    for _, row in v2.head(10).iterrows():
        dists = np.sqrt(
            (v3.latitude  - row.latitude )**2 * 111**2 +
            (v3.longitude - row.longitude)**2 *
             (np.cos(np.radians(row.latitude))*111)**2
        )
        cl  = v3.loc[dists.idxmin()]
        v2r = int(row[rank_col]) if rank_col else '?'
        print(f'  v2#{v2r:2d}: {row.latitude:.3f}N {abs(row.longitude):.3f}W  '
              f'v2={row[score_col]:.3f} → v3_rank={int(cl.v3_rank):2d} '
              f'v3={cl.v3_score:.3f}  tpi={cl.tpi:.1f}m  [{cl.submodel}]')
print()

print('=== SUMMARY ===')
print(f'North RF LOO  : {recall_n:.1%}  ({loo_n.correct.sum()}/{len(loo_n)})')
print(f'South RF LOO  : {recall_s_rf:.1%}  ({loo_s_rf.correct.sum()}/{len(loo_s_rf)})')
print(f'South SVM LOO : {recall_s_svm:.1%}  ({loo_s_svm.correct.sum()}/{len(loo_s_svm)})')
print(f'South selected: {south_type.upper()}')
print(f'v3 #1 : {v3.iloc[0].latitude:.4f}N  {v3.iloc[0].longitude:.4f}W  '
       f'score={v3.iloc[0].v3_score:.3f}  tpi={v3.iloc[0].tpi:.1f}m')


=== v2 TOP-10 → v3 (score=composite_score) ===
  v2#43: 40.823N 117.697W  v2=0.890 → v3_rank=36 v3=0.342  tpi=-6.6m  [north]
  v2# 6: 40.419N 117.704W  v2=0.880 → v3_rank=40 v3=0.243  tpi=7.3m  [north]
  v2# 1: 40.787N 117.456W  v2=0.853 → v3_rank= 8 v3=0.521  tpi=-0.0m  [north]
  v2#15: 40.405N 118.244W  v2=1.000 → v3_rank= 3 v3=0.608  tpi=18.2m  [north]
  v2#35: 40.265N 117.480W  v2=0.843 → v3_rank= 1 v3=0.689  tpi=-2.8m  [north]
  v2#17: 40.113N 117.202W  v2=0.849 → v3_rank=21 v3=0.457  tpi=-3.1m  [north]
  v2#19: 40.800N 118.140W  v2=0.797 → v3_rank=32 v3=0.371  tpi=0.4m  [north]
  v2#24: 40.096N 117.246W  v2=0.796 → v3_rank= 4 v3=0.595  tpi=3.9m  [north]
  v2#24: 40.096N 117.246W  v2=0.796 → v3_rank= 4 v3=0.595  tpi=3.9m  [north]
  v2#27: 40.209N 117.588W  v2=0.895 → v3_rank= 9 v3=0.517  tpi=-1.0m  [north]

=== SUMMARY ===
North RF LOO  : 65.2%  (15/23)
South RF LOO  : 71.4%  (5/7)
South SVM LOO : 0.0%  (0/7)
South selected: RF
v3 #1 : 40.2653N  -117.4797W  score=0.689  tpi=-2.8m


In [12]:
## 11. Save

joblib.dump(rf_north,     MODEL_DIR / 'paleowave_rf_v3_north.joblib')
joblib.dump(rf_south,     MODEL_DIR / 'paleowave_rf_v3_south.joblib')
joblib.dump(svm_south,    MODEL_DIR / 'paleowave_svm_v3_south.joblib')
joblib.dump(scaler_south, MODEL_DIR / 'paleowave_scaler_v3_south.joblib')

loo_n['submodel']     = 'north_rf'
loo_s_rf['submodel']  = 'south_rf'
loo_s_svm['submodel'] = 'south_svm'
pd.concat([loo_n, loo_s_rf, loo_s_svm]).to_csv(
    MODEL_DIR / 'paleowave_v3_loo_results.csv', index=False)

v3.head(50).to_csv(MODEL_DIR / 'paleowave_v3_top50.csv', index=False)
feats.to_csv(MODEL_DIR / 'paleowave_v3_training_features.csv', index=False)

print('Saved:')
for p in ['paleowave_rf_v3_north.joblib', 'paleowave_rf_v3_south.joblib',
          'paleowave_svm_v3_south.joblib', 'paleowave_scaler_v3_south.joblib',
          'paleowave_v3_loo_results.csv', 'paleowave_v3_top50.csv',
          'paleowave_v3_training_features.csv']:
    print(f'  {p}')
print()
print('=== PHASE 3 COMPLETE ===')
print(f'North RF  LOO: {recall_n:.1%}')
print(f'South {south_type.upper()} LOO: {south_recall:.1%}')


Saved:
  paleowave_rf_v3_north.joblib
  paleowave_rf_v3_south.joblib
  paleowave_svm_v3_south.joblib
  paleowave_scaler_v3_south.joblib
  paleowave_v3_loo_results.csv
  paleowave_v3_top50.csv
  paleowave_v3_training_features.csv

=== PHASE 3 COMPLETE ===
North RF  LOO: 65.2%
South RF LOO: 71.4%


In [14]:
print(v3[['v3_rank','latitude','longitude','v3_score','tpi','submodel']].head(10).to_string(index=False))

 v3_rank  latitude   longitude  v3_score       tpi submodel
       1 40.265278 -117.479722  0.688830 -2.762207    north
       2 40.907222 -118.464444  0.659855 -2.090210    north
       3 40.404722 -118.243889  0.607730 18.208130    north
       4 40.095556 -117.245556  0.594901  3.918579    north
       5 40.095556 -117.245556  0.594901  3.918579    north
       6 40.406389 -117.705556  0.553001  0.860840    north
       7 39.886111 -118.923333  0.537850  1.960327    north
       8 40.787222 -117.456389  0.521117 -0.014404    north
       9 40.209167 -117.587778  0.516667 -1.001831    north
      10 40.401111 -117.207778  0.515085  1.003296    north


In [15]:
# Distance from each v3 top-10 to nearest PBDB training point
known = feats[['latitude','longitude']].values

for _, row in v3.head(10).iterrows():
    dists_km = np.sqrt(
        (known[:,0] - row.latitude)**2  * 111**2 +
        (known[:,1] - row.longitude)**2 * (np.cos(np.radians(row.latitude))*111)**2
    )
    nearest_km = dists_km.min()
    nearest_idx = dists_km.argmin()
    nearest_occ = feats.iloc[nearest_idx].occurrence_id
    print(f'  v3#{int(row.v3_rank):2d}: {row.latitude:.3f}N {abs(row.longitude):.3f}W  '
          f'score={row.v3_score:.3f}  nearest_known={nearest_km:.1f}km  ({nearest_occ})')

  v3# 1: 40.265N 117.480W  score=0.689  nearest_known=13.8km  (occ:908661)
  v3# 2: 40.907N 118.464W  score=0.660  nearest_known=50.2km  (occ:906313)
  v3# 3: 40.405N 118.244W  score=0.608  nearest_known=9.1km  (occ:934002)
  v3# 4: 40.096N 117.246W  score=0.595  nearest_known=25.4km  (occ:909076)
  v3# 5: 40.096N 117.246W  score=0.595  nearest_known=25.4km  (occ:909076)
  v3# 6: 40.406N 117.706W  score=0.553  nearest_known=21.0km  (occ:908661)
  v3# 7: 39.886N 118.923W  score=0.538  nearest_known=59.8km  (occ:361674)
  v3# 8: 40.787N 117.456W  score=0.521  nearest_known=60.3km  (occ:906313)
  v3# 9: 40.209N 117.588W  score=0.517  nearest_known=4.3km  (occ:908661)
  v3#10: 40.401N 117.208W  score=0.515  nearest_known=41.0km  (occ:908661)


In [16]:
# Deduplicate v3 top50 and re-save
v3_clean = v3.drop_duplicates(subset=['latitude','longitude']).reset_index(drop=True)
v3_clean['v3_rank'] = v3_clean.index + 1
v3_clean.head(50).to_csv(MODEL_DIR / 'paleowave_v3_top50.csv', index=False)
print(f'Deduped: {len(v3)} → {len(v3_clean)} rows')
print(v3_clean[['v3_rank','latitude','longitude','v3_score','tpi']].head(10).to_string(index=False))

Deduped: 60 → 50 rows
 v3_rank  latitude   longitude  v3_score       tpi
       1 40.265278 -117.479722  0.688830 -2.762207
       2 40.907222 -118.464444  0.659855 -2.090210
       3 40.404722 -118.243889  0.607730 18.208130
       4 40.095556 -117.245556  0.594901  3.918579
       5 40.406389 -117.705556  0.553001  0.860840
       6 39.886111 -118.923333  0.537850  1.960327
       7 40.787222 -117.456389  0.521117 -0.014404
       8 40.209167 -117.587778  0.516667 -1.001831
       9 40.401111 -117.207778  0.515085  1.003296
      10 39.654722 -117.858889  0.503952 -3.083740
